# 04 · Releases and reruns: the inlist repository

POSYDON's physics lives in POSYDON-MESA-INLISTS. Since the restructure, a grid type's files are no longer found by rules in code but **listed in the repository itself**:

```
r11701/                              one folder per MESA version
  mesa/default/                      MESA's stock defaults/, columns/, src/
  ZAMS_models/
  releases/v2.0/
    release.ini                      MESA version and installation, frozen flag
    common/                          files shared by every grid type
    <grid type>/scenario.ini         the manifest: which files each target uses, in order
    reruns/<name>/scenario.ini       a physics variant: overlays and replacements
```

The code is `posydon/grids/setup/manifest.py` (reading releases) and `resolve.py` (getting the repository at the right version). The result of both is a `ScenarioFiles`: the `posydon_inlists` layer of the grid.

In [1]:
from nb_env import env, fresh, write, show, short, grid_csv
ENV = env()

import pandas as pd
from posydon.grids.setup.manifest import find_release, list_reruns, mesa_installation, release_files
from posydon.grids.setup.layers import LayerStack, drop_wiring_keys, fix_num_x_ctrls
from posydon.grids.setup.config import ConfigError

def rel(path):
    return short(path, ENV.release, ENV.inlists / "r11701")

## A release

In [2]:
release = find_release(ENV.inlists, "v2.0")
release

Release(name='v2.0', folder=PosixPath('/Users/max/Documents/POSYDON-MESA-INLISTS-v2-base/r11701/releases/v2.0'), mesa_version='r11701', mesa='default', frozen=True)

In [3]:
show(release.folder / "release.ini")

; POSYDON v2.0 inlists (MESA r11701)

[release]
name = v2.0
mesa_version = r11701
; the MESA installation, r11701/mesa/<name>/
mesa = default
frozen = true
; the inlist repository's main branch had this content at
source_commit = 9fca4599683fd214a7ac9d21e79f5ce2c0d74396
; and it equals production commit a060b7b


In [4]:
reruns = list_reruns(release)
print(len(reruns), "reruns:", ", ".join(reruns))

17 reruns: development_c4f90ce, fix_iniHe_pre_rerun5, jeff_new_grid, main_18710e9, matthias_LBV_wind, matthias_PISN, matthias_fix_initial_Hestar, matthias_lbv_thermo_dedt_noAgeLimit, matthias_no_age_limit, seth_TPAGBwind_runstar, seth_dedt_energy_eqn, seth_dedt_hepulse, seth_lbv_thermo_dedt, seth_lbv_thermo_dedt_hepulse, seth_lbv_thermo_dedt_hepulse_77640b5, seth_rerun_egrav_Pgas_dXdt_terms, zepei_fix_implicit


## A grid type's manifest

Each grid type has a `scenario.ini`. `[inlists]` lists, per target, the files applied in order (a later file wins key by key); `[columns]`, `[extras]` and `[zams]` name whole files.

In [5]:
show(release.folder / "CO-HeMS" / "scenario.ini")

; CO-HeMS grid type of release v2.0 (MESA r11701)
; Paths are relative to the release folder (releases/v2.0/);
; ZAMS models are relative to r11701/.

[grid_type]
name = CO-HeMS
description = a compact object and a helium star built in formation steps
star1 = formation
star2 = point_mass

[inlists]
; target = files, applied in order; a later file wins key by key.
; 'binary', 'star1' and 'star2' feed &controls and &star_job
; (&binary_controls and &binary_job); formation steps likewise.
binary = common/binary/inlist_project, CO-HeMS/binary/inlist_project
star1 = common/binary/inlist1, CO-HeMS/binary/inlist1
star2 = common/binary/inlist2
star1_formation_every_step = common/binary/inlist1
star1_formation_step0 = CO-HeMS/star1_formation/inlist_step1
star1_formation_step1 = CO-HeMS/star1_formation/inlist_step2

[columns]
history_columns = common/history_columns.list
binary_history_columns = common/binary_history_columns.list
profile_columns = common/profile_columns.list

[extras]
; legacy [

`release_files` reads it into the `ScenarioFiles` the rest of the setup uses. `binary = a, b` becomes the two targets `binary_controls` and `binary_job`, each with both files.

In [6]:
files = release_files(release, "CO-HeMS")
for target, paths in files.inlists.targets.items():
    print(f"{target:38} {', '.join(rel(p) for p in paths)}")
print("\ncolumn lists:", {k: rel(v) for k, v in files.inlists.whole_files.items()})
print("extras:      ", {k: rel(v) for k, v in files.extras.items()})
print("formed stars:", files.formation_stars, " ZAMS:", files.zams_star1, files.zams_star2)

binary_controls                        common/binary/inlist_project, CO-HeMS/binary/inlist_project
binary_job                             common/binary/inlist_project, CO-HeMS/binary/inlist_project
star1_controls                         common/binary/inlist1, CO-HeMS/binary/inlist1
star1_job                              common/binary/inlist1, CO-HeMS/binary/inlist1
star2_controls                         common/binary/inlist2
star2_job                              common/binary/inlist2
star1_formation_every_step_controls    common/binary/inlist1
star1_formation_every_step_job         common/binary/inlist1
star1_formation_step0_controls         CO-HeMS/star1_formation/inlist_step1
star1_formation_step0_job              CO-HeMS/star1_formation/inlist_step1
star1_formation_step1_controls         CO-HeMS/star1_formation/inlist_step2
star1_formation_step1_job              CO-HeMS/star1_formation/inlist_step2

column lists: {'history_columns': 'common/history_columns.list', 'binary_history_co

All six grid types side by side:

In [7]:
rows = []
for name in ["HMS-HMS", "CO-HMS_RLO", "CO-HeMS", "HeMS-HMS", "single_HMS", "single_HeMS"]:
    f = release_files(release, name)
    rows.append({"grid type": name,
                 "targets": len(f.inlists.targets),
                 "formation steps (star1)": sum(1 for t in f.inlists.targets if t.startswith("star1_formation_step") and t.endswith("_controls")),
                 "files": len({p for ps in f.inlists.targets.values() for p in ps}),
                 "own files": len({p for ps in f.inlists.targets.values() for p in ps if "common" not in str(p)}),
                 "formed": ", ".join(f.formation_stars) or "-",
                 "ZAMS in release": "yes" if (f.zams_star1 or f.zams_star2) else "no"})
pd.DataFrame(rows).set_index("grid type")

,targets,formation steps (star1),files,own files,formed,ZAMS in release
grid type,,,,,,
HMS-HMS,6,0,3,0,-,no
CO-HMS_RLO,6,0,6,3,-,no
CO-HeMS,12,2,7,4,star1,no
HeMS-HMS,12,2,6,3,star1,no
single_HMS,2,0,2,1,-,no
single_HeMS,8,3,4,3,star1,no


HMS-HMS and single_HMS use the common inlists only: their folder holds just a manifest (tracker C8/I5). Giving them files of their own is an edit of the manifest in the inlist repository, not of POSYDON.

## MESA's defaults come with the release

`release.ini` names the MESA installation (`mesa = default` → `r11701/mesa/default/`). Its `.defaults` files are the `mesa_inlists` layer of a new-format ini (notebook 05):

In [8]:
{k: rel(v) for k, v in mesa_installation(release).items()}

{'binary_controls': 'mesa/default/defaults/binary/binary_controls.defaults',
 'binary_job': 'mesa/default/defaults/binary/binary_job.defaults',
 'star_controls': 'mesa/default/defaults/star/controls.defaults',
 'star_job': 'mesa/default/defaults/star/star_job.defaults'}

## Reruns: physics variants

A rerun's folder mirrors the release and holds only what it changes. Its `scenario.ini` says how:

- `[overlays]`: an inlist with **only the changed keys**, applied right after the release file of the same path, wherever that file is used;
- `[replace]`: a whole file (a Fortran source) used instead of the release's.

In [9]:
rerun = "matthias_lbv_thermo_dedt_noAgeLimit"
show(release.folder / "reruns" / rerun / "scenario.ini")

; Rerun matthias_lbv_thermo_dedt_noAgeLimit of release v2.0 (MESA r11701)

[rerun]
name = matthias_lbv_thermo_dedt_noAgeLimit
release = v2.0
source_branch = matthias_lbv_thermo_dedt_noAgeLimit
source_commit = dcc42e1d6d89fc267826970643e7f4f1bf463171
; grid types the v2 production grids ran it for (ini files)
used_for = CO-HMS_RLO (24), CO-HeMS (8), HMS-HMS (67)

[overlays]
; release file = this rerun's file holding only the keys it
; changes; applied right after the release file, wherever
; the release file is used.
CO-HeMS/binary/inlist1 = CO-HeMS/binary/inlist1
common/binary/inlist1 = common/binary/inlist1
common/binary/inlist2 = common/binary/inlist2

[replace]
; release file = this rerun's file, used instead of it
CO-HeMS/star1_formation/src/run_star_extras.f = CO-HeMS/star1_formation/src/run_star_extras.f
common/binary/src/run_binary_extras.f = common/binary/src/run_binary_extras.f
common/binary/src/run_star_extras.f = common/binary/src/run_star_extras.f


In [10]:
show(release.folder / "reruns" / rerun / "common" / "binary" / "inlist1")

&controls
	max_age = -1
	thermohaline_coeff = 17.5d0
	surf_avg_tau_min = 10.0
	surf_avg_tau = 100.0
	surf_w_div_w_crit_limit = 0.95d0
	surf_w_div_w_crit_tol = 0.1d0
	mesh_Pgas_div_P_exponent = 0
	max_logT_for_k_below_const_q = 1.0d2
	max_q_for_k_below_const_q = 0.995d0
	min_q_for_k_below_const_q = 0.995d0
	max_logT_for_k_const_mass = 1.0d2
	max_q_for_k_const_mass = 0.99d0
	min_q_for_k_const_mass = 0.99d0
	x_logical_ctrl(5) = .true.
	x_logical_ctrl(6) = .false.
/ ! end of controls namelist


With the rerun, each overlay sits right after the file it changes:

In [11]:
with_rerun = release_files(release, "CO-HeMS", rerun)
for target in ["star1_controls", "star2_controls", "star1_formation_step0_controls"]:
    print(target)
    for p in with_rerun.inlists.targets[target]:
        print("   ", rel(p))
print("\nextras:", {k: rel(v) for k, v in with_rerun.extras.items()})

star1_controls
    common/binary/inlist1
    reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/inlist1
    CO-HeMS/binary/inlist1
    reruns/matthias_lbv_thermo_dedt_noAgeLimit/CO-HeMS/binary/inlist1
star2_controls
    common/binary/inlist2
    reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/inlist2
star1_formation_step0_controls
    CO-HeMS/star1_formation/inlist_step1

extras: {'posydon_binary_extras': 'reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/src/run_binary_extras.f', 'posydon_star_binary_extras': 'reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/src/run_star_extras.f', 'mesa_star1_extras': 'reruns/matthias_lbv_thermo_dedt_noAgeLimit/common/binary/src/run_star_extras.f'}


An unknown rerun lists the ones that exist:

In [12]:
try:
    release_files(release, "CO-HeMS", "matthias_typo")
except ConfigError as error:
    print(str(error)[:200], "...")

[scenario] rerun: no rerun 'matthias_typo' in release v2.0 (has: development_c4f90ce, fix_iniHe_pre_rerun5, jeff_new_grid, main_18710e9, matthias_LBV_wind, matthias_PISN, matthias_fix_initial_Hestar,  ...


## What does a variant change?

Because a rerun is an overlay, "what does this rerun change for this grid type?" is a small computation: merge the `posydon_inlists` files of each target with and without the rerun, and compare.

In [13]:
def clean(values):
    return fix_num_x_ctrls(drop_wiring_keys(values))

def merged_posydon(files):
    out = {}
    for target, paths in files.inlists.targets.items():
        stack = LayerStack(target)
        for p in paths:
            stack.add_file("posydon_inlists", p, transform=clean)
        out[target] = stack.merged()
    return out

def variant_changes(grid_type, rerun):
    """{target: {key: (release value, rerun value)}} and the replaced sources."""
    base = release_files(release, grid_type)
    variant = release_files(release, grid_type, rerun)
    a, b = merged_posydon(base), merged_posydon(variant)
    keys = {t: {k: (a[t].get(k), v) for k, v in b[t].items() if a[t].get(k) != v} for t in b}
    sources = {k: rel(v) for k, v in variant.extras.items() if base.extras.get(k) != v}
    return {t: c for t, c in keys.items() if c}, sources

keys, sources = variant_changes("CO-HeMS", rerun)
for target, changes in keys.items():
    print(target)
    for key, (old, new) in changes.items():
        print(f"    {key:40} {old!s:>14} -> {new}")
print("replaced sources:", sources)

star1_controls
    max_age                                          1.4e10 -> -1
    thermohaline_coeff                                0.0d0 -> 17.5d0
    surf_avg_tau_min                                    0.0 -> 10.0
    surf_avg_tau                                       10.0 -> 100.0
    surf_w_div_w_crit_limit                          0.99d0 -> 0.95d0
    surf_w_div_w_crit_tol                            0.05d0 -> 0.1d0
    mesh_Pgas_div_P_exponent                           None -> 0
    max_logT_for_k_below_const_q                       None -> 1.0d2
    max_q_for_k_below_const_q                          None -> 0.995d0
    min_q_for_k_below_const_q                          None -> 0.995d0
    max_logT_for_k_const_mass                          None -> 1.0d2
    max_q_for_k_const_mass                             None -> 0.99d0
    min_q_for_k_const_mass                             None -> 0.99d0
    x_logical_ctrl(5)                                  None -> .true.
    x_logical_ctrl

And for every rerun and grid type at once: the number of inlist keys it changes (summed over targets), and how many Fortran sources it replaces.

In [14]:
table = {}
for r in reruns:
    row = {}
    for gt in ["HMS-HMS", "CO-HMS_RLO", "CO-HeMS", "HeMS-HMS", "single_HMS", "single_HeMS"]:
        keys, sources = variant_changes(gt, r)
        n = sum(len(c) for c in keys.values())
        row[gt] = f"{n} keys" + (f" + {len(sources)} src" if sources else "")
    table[r] = row
pd.DataFrame(table).T

,HMS-HMS,CO-HMS_RLO,CO-HeMS,HeMS-HMS,single_HMS,single_HeMS
development_c4f90ce,2 keys + 3 src,2 keys + 3 src,3 keys + 3 src,3 keys + 3 src,1 keys + 3 src,1 keys + 3 src
fix_iniHe_pre_rerun5,5 keys + 3 src,5 keys + 3 src,8 keys + 3 src,7 keys + 3 src,3 keys + 3 src,3 keys + 3 src
jeff_new_grid,29 keys + 3 src,29 keys + 3 src,44 keys + 3 src,43 keys + 3 src,15 keys + 3 src,15 keys + 3 src
main_18710e9,0 keys,0 keys,1 keys,0 keys,0 keys,1 keys
matthias_LBV_wind,5 keys + 3 src,5 keys + 3 src,8 keys + 3 src,7 keys + 3 src,3 keys + 3 src,3 keys + 3 src
matthias_PISN,0 keys + 3 src,0 keys + 3 src,0 keys + 3 src,0 keys + 3 src,0 keys + 3 src,0 keys + 3 src
matthias_fix_initial_Hestar,29 keys + 3 src,29 keys + 3 src,44 keys + 3 src,43 keys + 3 src,15 keys + 3 src,15 keys + 3 src
matthias_lbv_thermo_dedt_noAgeLimit,29 keys + 3 src,29 keys + 3 src,44 keys + 3 src,43 keys + 3 src,15 keys + 3 src,15 keys + 3 src
matthias_no_age_limit,7 keys + 3 src,7 keys + 3 src,11 keys + 3 src,10 keys + 3 src,4 keys + 3 src,4 keys + 3 src
seth_TPAGBwind_runstar,2 keys + 3 src,2 keys + 3 src,3 keys + 3 src,3 keys + 3 src,1 keys + 3 src,1 keys + 3 src


`[rerun] used_for` in each rerun's `scenario.ini` records which grid types the v2 production grids actually ran it for.

## Getting the repository at the right version

`resolve.resolve_scenario(config)` turns the ini's `[scenario]` into a `ResolvedScenario`: the files, the repository folder and the **commit** they came from (written to `setup_record.ini`).

- `ref = <branch or tag>` or `commit = <hash>`: the repository (by default the clone `~/.posydon_mesa_inlists`, or `inlist_repo`) is fetched and checked out, detached, at that version. A ref or commit that is not there is an error; nothing falls back to "whatever is checked out".
- `source = path`: the folder in `inlist_repo` is used as it is, without git; handy while editing inlists. If you also give `commit`, it checks that the folder is at that commit.
- A legacy scenario (`['posydon', 'branch-commit', 'system type']`) checks out that old commit and finds the files by the old rules (`scenario.scenario_files`).

In [15]:
from posydon.grids.setup.config import read_config
from posydon.grids.setup.resolve import resolve_scenario

folder = fresh("04_resolve")
grid_csv(folder)
config = read_config(write(folder / "grid.ini", f"""
    [grid]
    type = CO-HeMS
    points = grid.csv
    [scenario]
    release = v2.0
    rerun = {rerun}
    source = path
    inlist_repo = {ENV.inlists}
    """))
resolved = resolve_scenario(config)
print("repo:   ", resolved.repo)
print("commit: ", resolved.commit)
print("release:", resolved.release.name, resolved.release.mesa_version, resolved.release.mesa)
print("same files as release_files:", resolved.files.inlists.targets == with_rerun.inlists.targets)

repo:    /Users/max/Documents/POSYDON-MESA-INLISTS-v2-base
commit:  115de83c33bad5013cb09837fcc679d1d45e816b
release: v2.0 r11701 default
same files as release_files: True


## Try it

- Pick the rerun with the most changes in the table and look at its overlay files.
- Make a rerun of your own: copy the inlist repository, add `releases/v2.0/reruns/my_test/scenario.ini` with one overlay that changes one key, point `inlist_repo` at the copy and run `variant_changes("HMS-HMS", "my_test")`.
- Run `variant_changes` for `seth_dedt_energy_eqn` and compare with `seth_lbv_thermo_dedt`.